# Spark
- Unified computing engine. 
- Not made for storage. 
- Storage is separated and can be could storage or on prem storage

- Advantages of spark over Hadoop
  - Spark SQL and SQL engine is faster
  - Support for more languages: Java, Scala, Python, R, SQL (Scala is native language of Spark)
  - Storage is separated from compute, no need to depend on any storage technology like Hadoop is dependent on HDFS
  - Streaming and ML support through libraries Note: Spark on Hadoop/ADLS is Data Lake, Spark on Databricks is Lakehouse

## Spark Architecture
- Storage Layes: Decoupled - Azure, AWS, GCP, HDFS - Can choose from any available options
- Compute Hadware (CPUs/ RAM-Memory) - To be used as worker-executor nodes for compute
- Cluster Manager - YARN, Kubernaties, Mesos etc. can be used
- Spark Engine
- Spark Core (Scala, Java, Python, R)
- Higher APIs/Libraries: Spark SQL Dataframes, Streaming, MLib, GraphX


https://spark.apache.org/docs/latest/cluster-overview.html

![image_1783749090578.png](./image_1783749090578.png "image_1783749090578.png")



Cluster --> (Driver - Work architecture)

![image_1783748188108.png](./image_1783748188108.png "image_1783748188108.png")

## Spark Deployment (How and where can I use Spark?)


- _Local Spark installation:_ You can install Apache Spark directly on your laptop or VM.
- _Standalone Spark cluster:_ Spark has its own built-in cluster manager called Standalone Mode.
- Spark on available _cluster managers in market_ (YARN, Kubernates)
- _Platform as a Service:_ Databricks/ Azure Synapse Spark/AWS EMR

## Spark Concepts
- Spark Application
- Spark Session
- Spark Context
- Driver
- Cluster Manager
- Executor (Worker)

- Spark application is a set of spark commands written to transform the data to be executed as a bundle.
- User writes a Spark application then submit it to driver for actually processing the data using tool called Spark Submit (Or in case of databricks notebook by simpling running cell).
- Spark application communicates to driver using spark session. Spark session is an entry point to programing spark dataframe and dataset API

**Note:** Since PySpark 2.0, Creating a SparkSession creates a SparkContext internally and exposes the sparkContext variable to use. Before Spark 2.0, SparkContext used to be an entry point, and it’s not been completely replaced with SparkSession. Many features of SparkContext are still available and used in Spark 2.0 and later. You should also know that SparkSession internally creates SparkConfig and SparkContext with the configuration provided with SparkSession. Sparkcontext can be set to Spark, SQL, Streaming, Hive etc. But now it all merged to Spark context. SparkSession is a higher-level API that's easier to use, while SparkContext is a low-level API that's used for more control. SparkSession internally owns SparkContext. SparkContext is the low-level execution engine responsible for communicating with the cluster manager and managing RDD execution. SparkSession is the unified, high-level entry point introduced in Spark 2.x that encapsulates SparkContext, SQLContext, and HiveContext and is used for DataFrame, SQL, and streaming operations.

- Driver divides task between Executors using cluster manager and collects results to show the results to user.

**Note:** In databricks notebook, there is no need to create a SparkSession, it is created by default. In Databricks, both SparkContext and SparkSession are auto-created when a notebook is attached to a cluster. Databricks initializes the SparkContext first, then creates a SparkSession and exposes it as the global variable spark, while sc refers to the underlying SparkContext.

**Lifecycle in databricks**

- Notebook Attach
↓
- Cluster Starts
↓
- Driver JVM Created
↓
- SparkContext Initialized
↓
- SparkSession Initialized
↓
- Variables spark & sc Injected
↓
- User Code Executes

In [0]:
# Create SparkSession from builder
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[1]") \
                .appName('SparkByExamples.com') \
                .getOrCreate()
print(spark.sparkContext)
print("Spark App Name : "+ spark.sparkContext.appName)

# Outputs
#<SparkContext master=local[1] appName=SparkByExamples.com>
#Spark App Name : SparkByExamples.com

## DAG (Directed Acyclic Graph)
- The logical execution plan for a data processing job
- Lazy evalution: trasformations are not executed right a way. DAG is created after each trasformation but not executed
- Once action is called, DAG is used for executing all trasformation requred before including action

![image_1783749795186.png](./image_1783749795186.png "image_1783749795186.png")

## Spark Application, Job, Stages, Task

- For every data processing request in Apache spark (for each application), jobs are created and submitted to resource manager
- Each job has multiple stages and each stage have multiple tasks
- Each action corresponds to one Job (1 Action = 1 Job)


**Spark APIs**
- RDD (Resilient Distributed Datase)
- Dataset
- Dataframe
- SQL

## Spark SQL Engine (AKA Catalyst Optimizer)

- SQL/Dataframe API/Dataset related application code is submitted to Spark SQL engine
- Spark SQL engine converts it into java bytecodes generating physical plan
- Four phases of Spark SQL engine
  - Analysis
  - Logical Planning
  - Physical Planning
  - Code generation
  - spark-sql-engine

![image_1783749564345.png](./image_1783749564345.png "image_1783749564345.png")

## Trasnformations vs Actions
- **Trasformations:** Functions applied on dataframes or RDD which generates new DF or RDD by creating DAG but do not execute till action function is pplied and called
- **Actions:** Functions applied on dataframe that causes execution of job based on DAG created by trasformations

## Narrow vs Wide (Dependency) transfroamtions
- For distributed system, data is saved on different partitions on file system (like HDFS)
- Lets say we have a file/dataframe of 500 MB and default partition size of storage (e.g HDFS) is 128 MB then data will be saved in total (500/128 = ) 4 partitions
- For any trasformation function applied if shuffle happens between partitions then it is considered as Wide trasformation otherwise it is narrow trasformation
- If data is save in single partition then there will be no re shuffling on data and usual wide trasformations will act as a narrow trasformation
- Wide trasformations are expensive and should be avoided if possible
- Examples of wide and narrow transformations

|Narrow Trasformations|Wide Trasformations|
|--|--|
|filter|groupby|
|select|join|
|union|distinct|


## Repartition vs Coalesce
- Data skewness: Unevenly distributed data across partitions
- To avoid skewness and make data distributed evenly in partitions use repartition or Coalesce
df.repartition(5) or df.Coalesce(3)
- Repartition:
  - Can increase or decrease number of existing paritions.
  - Data is evenly distributed after repartition
  - Shuffle happens
  - Expensive operaration when shuffle happens
- Coalesce:
  - Only merge smaller partitions into larger one
  - Since only merge happens number of partitions are only reduced
  - No shuffle, so inexpensive

## Types of Spark physical Joins
- Shuffle sort-merge join
- Shuffle hash join
- Broadcast hash join
- Cartesian join
- Boradcast nested loop join


|Join Strategy|	Short Meaning|	Best Used When|
|--|--|--|
|Shuffle Sort-Merge Join|	Both datasets are shuffled, sorted by join key, then merged	|Large datasets on both sides|
|Shuffle Hash Join|	Both datasets are shuffled, then one side is hashed per partition	One side is |smaller but not broadcasted|
|Broadcast Hash Join|	Small table is broadcast to all executors and hashed|	One table is small|
|Cartesian Join|	Every row from left joins with every row from right	|Rare cases, cross join|
|Broadcast Nested Loop Join|	Broadcasts one side and compares every row using nested loops	Non-equi |joins or cross joins with small table|

- If both tables are large, Spark commonly uses Shuffle Sort-Merge Join.
- If one table is small, Spark can use Broadcast Hash Join.
- For non-equality joins or cross joins, Spark may use Broadcast Nested Loop Join or Cartesian Join.

Example:

```python

broadcast_hash_join_df = large_left.join(
    broadcast(small_right),
    on="key",
    how="inner"
)

broadcast_hash_join_df.show(5)
broadcast_hash_join_df.explain(True)
```

## Driver OOM Exception
- When driver is requested with below configuration, continer of 11 GB is created for a driver. 1 GB space of memory overhead is used for container creation (30-400 MB)and management and for some non-JVM processes.
```python
    spark.driver.memory = 10G
    spark.driver.memoryoverhead = 1G
```
- While performing operations/trasformations when driver memory (10G or 1G in above case) is full, out-Of-Memory exception is thrown
- Memory overhead: 10% or RAM or 384 MB whichever is higher (Used for non-JVM process + continer managetment process)
- Possible reasons: collect() action, broadcast join, container limit reached beyond allocated space for memory overhead etc

## Executor OOM Exception
- While performing operations/trasformations when executor memory is full (10G or 1G in above case) is full, Oout-Of-Memory exception is thrown
- Executor memory is divided into three portions
  - Reserved memory (e.g 300 MB)
    - used for Spark internal objects
    - Total executor memory should be greater than 1.5X of reserved memory
  - Spark memory (e.g. 60% of remaining)
    - Storage memory pool
    - Executor memory pool
  - User Memory (e.g 40% of remaining)
    - Used for user data structures
- Memory manage
  - static
  - unified

## AQE (Adaptive Query Execution)
- Adaptive Query Execution (AQE) is a runtime optimization framework in Apache Spark that adapts the physical execution plan based on actual runtime statistics (shuffle sizes, skew, etc.). 
- It lets Spark change join strategies, coalesce shuffle partitions, and mitigate skew during execution to improve performance.
- Main benefits:
  - Choose a better join strategy (e.g., switch to broadcast) based on actual sizes.
  - Coalesce/reduce the number of post-shuffle partitions to avoid many small tasks.
  - Detect skewed partitions and split them to avoid overloaded tasks.

**Use AQE when:**
- Data statistics at planning time are unreliable or data sizes are dynamic.

**Disable AQE when:**
- You need fully deterministic, repeatable plans for debugging or benchmarking.
- You rely on a specific physical plan or custom tuning and want to avoid plan changes at runtime.